# Session 3 Update + Config Corrections (2026-03-31)

This notebook covers two things we updated after the initial builds:

1. **`utils/timer.py`** — `calculate_cost()` updated to support tiered pricing
2. **`core/config.py`** — model catalogue overhauled: GPT-5.4 series, Gemini 2.5 only, corrected Claude pricing

---

## Why tiered pricing?

Some providers charge **more per token when your context window is large.**
The idea is: longer contexts are more expensive to process on their end, so they pass that cost on.

| Model | Threshold | Input below | Input above |
|---|---|---|---|
| Gemini 2.5 Pro | 200K tokens | $1.25/1M | $2.50/1M |
| GPT-5.4 | 128K tokens | $2.50/1M | $5.00/1M |
| GPT-5.4 Pro | 128K tokens | $30.00/1M | $60.00/1M |
| Claude (all) | — | flat rate | same flat rate |

Claude explicitly documents that its **full 1M context window is billed at the same flat rate** — no tier penalty.

---

## New fields in `ModelConfig`

In [ ]:
from dataclasses import dataclass, field

@dataclass(frozen=True)
class ModelConfig:
    provider: str
    model_id: str
    display_name: str
    input_cost_per_1k: float
    output_cost_per_1k: float
    supports_vision: bool = field(default=False)
    # --- NEW: tiered pricing fields ---
    large_context_threshold: int   = field(default=0)    # 0 = no tiered pricing
    input_cost_per_1k_large: float = field(default=0.0)  # rate above threshold
    output_cost_per_1k_large: float = field(default=0.0) # rate above threshold

**Why `default=0` for the threshold?**  
If `large_context_threshold` is 0, no model's input will ever exceed it (tokens are always > 0),
so the large-context branch never fires. Models without tiers simply omit these fields and pay flat rates.

This design means **existing code doesn't break** when you add these fields — models that don't
set them just use the standard rates. Zero is the right default here because it acts as a sentinel
meaning "no tiered pricing."

---

## Updated `calculate_cost()`

In [ ]:
def calculate_cost(model: ModelConfig, input_tokens: int, output_tokens: int) -> float:
    # Determine which tier applies
    use_large = (
        model.large_context_threshold > 0          # model has tiered pricing
        and input_tokens > model.large_context_threshold  # request exceeds threshold
    )
    input_rate  = model.input_cost_per_1k_large  if use_large else model.input_cost_per_1k
    output_rate = model.output_cost_per_1k_large if use_large else model.output_cost_per_1k

    input_cost  = (input_tokens  / 1000) * input_rate
    output_cost = (output_tokens / 1000) * output_rate
    return round(input_cost + output_cost, 6)

The logic is a simple two-step check:
1. Does this model even have tiered pricing? (`threshold > 0`)
2. If yes, did the request exceed the threshold? (`input_tokens > threshold`)

Only if both are true do we use the large-context rates. Otherwise, standard rates apply.

In [ ]:
# Worked example — Gemini 2.5 Pro, same prompt at two different context sizes

gemini_pro = ModelConfig(
    provider="gemini", model_id="gemini-2.5-pro", display_name="Gemini 2.5 Pro",
    input_cost_per_1k=0.001250,        # $1.25/1M  ≤200K
    output_cost_per_1k=0.010000,       # $10.00/1M ≤200K
    large_context_threshold=200_000,
    input_cost_per_1k_large=0.002500,  # $2.50/1M  >200K
    output_cost_per_1k_large=0.015000, # $15.00/1M >200K
)

# Short context — 500 input, 200 output (well under 200K)
short = calculate_cost(gemini_pro, input_tokens=500, output_tokens=200)
print(f"Short context: ${short:.6f}")
# (500/1000)*0.00125 + (200/1000)*0.010 = 0.000625 + 0.002 = $0.002625

# Long context — 250K input, 500 output (over 200K threshold)
long = calculate_cost(gemini_pro, input_tokens=250_000, output_tokens=500)
print(f"Long context:  ${long:.6f}")
# (250000/1000)*0.0025 + (500/1000)*0.015 = 0.625 + 0.0075 = $0.632500

---

## Model catalogue overhaul

### Why we replaced GPT-4.1 with GPT-5.4

The GPT-5.4 family is the current generation. We kept the same four-tier structure
(Pro → standard → Mini → Nano) but updated all model IDs and prices.

GPT-5.4 and GPT-5.4 Pro have short/long context pricing. Mini and Nano do not.

**Note:** OpenAI's pricing page does not document the exact token threshold for the short/long
boundary — we used 128K as a reasonable default. Update `large_context_threshold` in `config.py`
if OpenAI publishes the exact number.

### Why we replaced Gemini 2.0 with Gemini 2.5 only

During testing, we hit:
```
404 NOT_FOUND: This model models/gemini-2.0-flash-lite is no longer available
```
Google deprecated the 2.0 models. The 2.5 family (Pro, Flash, Flash Lite) is the current active generation.

### Why we also migrated the Gemini SDK

The old `google-generativeai` package printed a deprecation warning on every import:
```
FutureWarning: All support for the google.generativeai package has ended.
```
We swapped it for the new `google-genai` package and rewrote `gemini_client.py` to use
`client.aio.models.generate_content(...)` with typed `Content`/`Part` objects.

### Claude pricing correction

The original config had Opus 4.6 at **$15/$75** — that was actually Opus 4.1 pricing.
Haiku 4.5 was at **$0.80/$4** — that was Haiku 3.5 pricing.

Corrected values (from platform.claude.com/docs/en/about-claude/pricing):

| Model | Old Input | New Input | Old Output | New Output |
|---|---|---|---|---|
| Claude Opus 4.6 | $15/1M | **$5/1M** | $75/1M | **$25/1M** |
| Claude Sonnet 4.6 | $3/1M | $3/1M ✓ | $15/1M | $15/1M ✓ |
| Claude Haiku 4.5 | $0.80/1M | **$1/1M** | $4/1M | **$5/1M** |

Lesson: always verify pricing against the official docs page before shipping.

---

## Summary

| What changed | Why |
|---|---|
| `ModelConfig` got 3 new fields for tiered pricing | Gemini 2.5 Pro and GPT-5.4/Pro charge more above a token threshold |
| `calculate_cost()` selects rate based on threshold | Automatically applies the right tier without changing call sites |
| Claude: no tiered pricing | Anthropic explicitly documents flat pricing across the full 1M window |
| Gemini SDK: `google-generativeai` → `google-genai` | Old package officially deprecated |
| Gemini models: 2.0 removed, 2.5 only | 2.0 models return 404 — no longer available |
| OpenAI models: GPT-4.1 → GPT-5.4 series | Current generation |
| Claude Opus 4.6 price: $15 → $5 input | Previous config had Opus 4.1 pricing by mistake |
| Claude Haiku 4.5 price: $0.80 → $1 input | Previous config had Haiku 3.5 pricing by mistake |